# Práctica B: ¿Puede la IA predecir la cosecha de café? Jaén (Cajamarca)

Este notebook implementa el **Caso 2** de la investigación con datos 100% auditados y reproducibles:
- **24 campañas agrícolas continuas (2001–2024)** en la cuenca cafetalera de Jaén (Cajamarca, Perú).
- **Rendimiento oficial de café** extraído de compendios estadísticos del **INEI** y **MIDAGRI-SIEA** (2010 a 2024).
- **Sensoramiento Remoto MODIS MOD13Q1 (250m, 16 días)**: NDVI y EVI anual y en ventanas fenológicas (Floración: Ago-Oct; Llenado: Nov-Feb).
- **Clima diario ERA5-Land**: Precipitación, balance hídrico y temperaturas extremas.
- **Benchmark riguroso de 16 modelos** evaluados bajo **Walk-Forward expansivo (Leave-Future-Out)** de 2016 a 2024.
- Auditoría de las 4 hipótesis científicas (H1: saturación de NDVI, H2: desacople fenológico, H3: ventaja multimodal, H4: ceguera espectral de la roya 2012-2014).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

df = pd.read_csv('../data/caso_2/cafe_jaen_campanias.csv')
print(f'Total campañas históricas: {len(df)}')
con_rend = df.dropna(subset=['rendimiento_qq_ha']).reset_index(drop=True)
display(con_rend[['campania', 'rendimiento_qq_ha', 'superficie_cosechada_ha', 'produccion_t', 'ndvi_anual', 'evi_flor', 'lluvia_flor_mm', 'def_hidrico_flor_mm']])

## 1. La Trampa de la Saturación (H1 y H4): Rendimiento vs Satélite Óptico
Obsérvese cómo el NDVI promedio anual permanece prácticamente plano en ~0.74 (CV = 2.3%). Durante la epidemia de la roya amarilla (2011–2014), el rendimiento oficial colapsó un **-56.0%** (de 22.7 a 10.0 qq/ha), pero el satélite óptico aumentó **+1.4%** debido al dosel de árboles de sombra y malezas de sotobosque.

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5))
color = '#0d9488'
ax1.set_xlabel('Campaña Agrícola', fontweight='bold')
ax1.set_ylabel('Rendimiento (qq pergamino / ha)', color=color, fontweight='bold')
line1 = ax1.plot(con_rend['campania'], con_rend['rendimiento_qq_ha'], 'o-', color=color, linewidth=2.5, label='Rendimiento Oficial INEI')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, linestyle='--', alpha=0.5)

ax2 = ax1.twinx()
color_sat = '#16a34a'
ax2.set_ylabel('MODIS NDVI Anual (250m)', color=color_sat, fontweight='bold')
line2 = ax2.plot(con_rend['campania'], con_rend['ndvi_anual'], 's--', color=color_sat, linewidth=2, label='NDVI Anual (Satélite)')
ax2.set_ylim(0.5, 0.9)
ax2.tick_params(axis='y', labelcolor=color_sat)
ax2.grid(False)

# Marcador de la Crisis de la Roya
ax1.axvspan(2012, 2014, color='#ef4444', alpha=0.15, label='Crisis de la Roya Amarilla (2012-2014)')

lines = line1 + line2
labels = [l.get_label() for l in lines] + ['Crisis de la Roya']
ax1.legend(lines, labels, loc='upper left')
plt.title('Caso 2: Quiebre Estructural y Ceguera Espectral del Satélite Óptico', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 2. Benchmark de 16 Modelos en Validación Walk-Forward (Leave-Future-Out)
Comparamos 16 modelos entrenados sin fuga temporal (8 campañas de test ciego: 2016, 2018–2024). Notemos cómo los modelos de clima fenológico y árboles parsimoniosos superan por más de 30% al modelo de visión satelital pura (Solo NDVI).

In [ ]:
df_bm = pd.read_csv('../data/caso_2/benchmark_cafe.csv')
print('=== BENCHMARK DE 16 MODELOS EN CAFÉ JAÉN ===')
display(df_bm[['modelo', 'familia', 'mae_walkforward', 'rmse_walkforward', 'mape_walkforward', 'error_abs_2020', 'p_valor_vs_mejor']])

fig, ax = plt.subplots(figsize=(11, 7))
df_sorted = df_bm.sort_values('mae_walkforward', ascending=False).reset_index(drop=True)
colors = ['#059669' if p == 1.0 else '#ef4444' if 'Sensoramiento' in fam else '#0284c7' if 'Clima' in fam else '#10b981' for p, fam in zip(df_sorted['p_valor_vs_mejor'], df_sorted['familia'])]
bars = ax.barh(df_sorted['modelo'], df_sorted['mae_walkforward'], color=colors, height=0.65, edgecolor='#1e293b')
ax.set_xlabel('MAE en Validación Walk-Forward (qq/ha) — Menor es mejor', fontweight='bold')
ax.set_title('Benchmark de 16 Modelos en Café Jaén: Clima Fenológico vs Satélite Óptico', fontweight='bold', pad=12)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.05, bar.get_y() + bar.get_height()/2, f'{w:.2f} qq/ha', va='center', fontsize=8.5, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Trayectorias de Pronóstico Walk-Forward Año por Año (2016–2024)
Comparamos la respuesta de los modelos ante el récord histórico de 2020 (35.38 qq/ha) y la estabilidad en campañas normales.

In [ ]:
df_preds = pd.read_csv('../data/caso_2/cafe_jaen_predicciones_modelos.csv')
plt.figure(figsize=(11, 5.5))
plt.plot(df_preds['campania'], df_preds['rendimiento_real'], 'k*-', linewidth=3, markersize=10, label='Observado Oficial (INEI)')
plt.plot(df_preds['campania'], df_preds['M12: Random Forest Parsimonioso (d=2)'], 'g-o', linewidth=2.2, label='M12: Random Forest (Ganador MAE 3.72)')
plt.plot(df_preds['campania'], df_preds['M08: Solo Déficit Hídrico Floración'], 'b--s', linewidth=2.0, label='M08: Solo Déficit Hídrico (MAE 3.90)')
plt.plot(df_preds['campania'], df_preds['M11: ElasticNet Multimodal (L1+L2)'], 'm-.^', linewidth=1.8, label='M11: ElasticNet (MAE 4.33)')
plt.plot(df_preds['campania'], df_preds['M05: Solo NDVI Anual (Saturado)'], 'r:d', linewidth=2.2, label='M05: Solo NDVI Anual (Falla MAE 5.45)')
plt.title('Validación Walk-Forward en Jaén (2016–2024): Trayectorias de Pronóstico Out-of-Sample', fontweight='bold')
plt.xlabel('Campaña Agrícola', fontweight='bold')
plt.ylabel('Rendimiento (qq/ha)', fontweight='bold')
plt.xticks(df_preds['campania'])
plt.legend(ncol=2)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Auditoría Científica de las 4 Hipótesis (H1, H2, H3, H4)

In [ ]:
with open('../data/caso_2/cafe_jaen_hipotesis.json', 'r', encoding='utf-8') as f:
    hip = json.load(f)
for h_key, h_val in hip.items():
    print(f'=== {h_key.upper()} ({h_val["estado"]}) ===')
    print(f'Premisa: {h_val["hipotesis"]}')
    print(f'Interpretación: {h_val["interpretacion"]}\n')